# Aurora Siger — Sistema de Verificação de Pré-Decolagem

Sistema de telemetria e checklist automatizado para a **Missão Aurora Siger — Ignition Zero, Fase 1**.

Antes de qualquer tentativa de lançamento, todos os parâmetros críticos da nave são verificados sequencialmente. Ao encontrar a primeira falha, o sistema aborta imediatamente e informa o motivo — o mesmo comportamento de um sistema real de pré-lançamento. Se todos os parâmetros estiverem dentro dos limites operacionais, a análise é encaminhada ao modelo de IA para uma avaliação adicional de risco.

Os limites utilizados neste notebook têm base no **Relatório de Análise Energética e Propulsiva da Missão Aurora Siger (Março de 2026)** e nas referências técnicas da NASA, PCB Aerospace, TE Connectivity e Collins Aerospace listadas naquele documento.

---

> **Antes de executar:** Configure sua chave da API Gemini em **Secrets** (ícone 🔑 no painel esquerdo do Colab) com o nome exato `GEMINI_API_KEY`. Sem ela, a célula 2 retornará erro.

## 1. Instalação

Instala o SDK oficial do Google para acesso à API Gemini.

O pacote `google-genai` é a biblioteca Python mantida pelo Google que permite enviar prompts e receber respostas dos modelos Gemini diretamente pelo código. A flag `-q` suprime as mensagens de progresso do pip para manter a saída limpa.

> Esta célula só precisa ser executada uma vez por sessão do Colab. Se você reiniciar o ambiente de execução, execute-a novamente antes das demais.

In [ ]:
!pip install -q google-genai

## 2. Configuração do cliente Gemini

Lê a chave de API dos **Secrets** do Colab e inicializa o cliente de comunicação com o Gemini.

O Colab oferece um cofre seguro de variáveis chamado **Secrets**, que permite armazenar chaves de API sem precisar colá-las diretamente no código. A função `userdata.get('GEMINI_API_KEY')` busca o valor pelo nome configurado no painel de Secrets.

O objeto `cliente` criado aqui é o ponto de entrada para todas as chamadas à API — ele será reutilizado na célula 5 para enviar o prompt de análise de telemetria.

> Se esta célula retornar `SecretNotFoundError`, verifique se o nome do secret está exatamente como `GEMINI_API_KEY` (sem espaços, maiúsculas conforme indicado) e se o toggle de acesso ao notebook está ativado.

In [ ]:
from google.colab import userdata
from google import genai

cliente = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))
print('Cliente Gemini configurado.')

## 3. Dados de telemetria

Insira os valores de telemetria da nave quando solicitado. Cada `input()` aguarda você digitar o número e pressionar `Enter`.

Os limites abaixo foram definidos com base no **Relatório de Análise Energética e Propulsiva da Missão Aurora Siger** e nas referências técnicas da NASA, PCB Aerospace e Collins Aerospace. As pressões dos tanques têm faixas diferentes porque o RP-1 (combustível) e o LOX (oxidante) operam em regimes distintos — o LOX precisa de pressão maior para garantir o fluxo adequado dado que a razão de mistura é de 2 partes de LOX para 1 de RP-1.

| Parâmetro | Faixa válida | Origem do limite |
|---|---|---|
| Temperatura interna (eletrônicos) | -10°C a 40°C | Limites operacionais dos sensores NTC e ECUs embarcados |
| Temperatura externa (estrutura) | -5°C a 45°C | Limites estruturais do invólucro da nave |
| Energia das baterias | ≥ 70% | Margem mínima para operar todos os sistemas críticos durante as 3 fases |
| Combustível RP-1 | ≥ 80% | Cenário B do relatório exige 95% de propelente — 80% é o piso de alerta |
| Oxidante LOX | ≥ 80% | Mesma lógica do RP-1; razão de mistura 1:2 (RP-1:LOX) |
| Pressão do tanque de RP-1 | 200 a 400 kPa | Faixa operacional dos sensores piezoelétricos ICP® (PCB Aerospace) |
| Pressão do tanque de LOX | 300 a 500 kPa | LOX requer pressurização maior para compensar densidade e vazão |
| Integridade estrutural | 1 = OK, 0 = ERRO | Leitura binária dos sensores estruturais da nave |

In [ ]:
# INPUTS
temp_interna       = float(input('Temperatura interna (-10 a 40°C): '))
temp_externa       = float(input('Temperatura externa (-5 a 45°C): '))
nivel_energia      = float(input('Nível de energia (% mínimo 70): '))
nivel_combustivel  = float(input('Nível de combustível RP-1 (% mínimo 80): '))
nivel_oxidante     = float(input('Nível de oxidante LOX (% mínimo 80): '))
pressao_tanque_rp1 = float(input('Pressão do tanque de RP-1 (200 a 400 kPa): '))
pressao_tanque_lox = float(input('Pressão do tanque de LOX (300 a 500 kPa): '))
integridade        = bool(input('Integridade estrutural (1 = OK, 0 = ERRO): '))

## 4. Verificação de sistemas

Valida cada parâmetro sequencialmente e exibe o resultado linha a linha.

**Como funciona a lógica de verificação:**

Antes das comparações diretas, o código calcula o **gradiente térmico** — a diferença absoluta entre temperatura interna e externa. Esse valor não é lido por sensor, mas derivado dos dois inputs de temperatura. Um gradiente acima de 40°C indica que a estrutura e os eletrônicos estão em regimes térmicos incompatíveis, o que pode causar dilatação diferencial e falha de componentes.

Cada verificação gera uma tupla `(nome, passou, mensagem_de_erro)` adicionada à lista `resultados`. Ao final, o `for/else` percorre essa lista:
- Se encontrar `passou = False`, imprime a falha e executa `break` — **o laço para na primeira falha**.
- Se percorrer todos os itens sem `break`, o bloco `else` executa — comportamento exclusivo do `for/else` em Python — e imprime ✅ PRONTO PARA DECOLAR.

O `time.sleep(0.7)` entre cada linha de OK é intencional: simula o tempo real de leitura e validação de cada subsistema, tornando a saída mais legível e próxima de um checklist real de missão.

In [ ]:
import time

# CÁLCULOS
gradiente_termico = abs(temp_interna - temp_externa)

print('\n[ Analisando sistema... ]\n')
time.sleep(0.5)

# VERIFICAÇÕES
resultados = []

if not (-10 <= temp_interna <= 40):
    resultados.append(('Temperatura interna', False, f'Fora do limite ({temp_interna}°C). Esperado: -10°C a 40°C.'))
else:
    resultados.append(('Temperatura interna', True, None))

if not (-5 <= temp_externa <= 45):
    resultados.append(('Temperatura externa', False, f'Fora do limite ({temp_externa}°C). Esperado: -5°C a 45°C.'))
else:
    resultados.append(('Temperatura externa', True, None))

if not (gradiente_termico <= 40):
    resultados.append(('Gradiente térmico', False, f'Gradiente excessivo ({gradiente_termico:.1f}°C). Máx: 40°C.'))
else:
    resultados.append(('Gradiente térmico', True, None))

if not (nivel_energia >= 70):
    resultados.append(('Nível de energia', False, f'Energia insuficiente ({nivel_energia:.1f}%). Mín: 70%.'))
else:
    resultados.append(('Nível de energia', True, None))

if not (nivel_combustivel >= 80):
    resultados.append(('Nível de combustível (RP-1)', False, f'Combustível insuficiente ({nivel_combustivel:.1f}%). Mín: 80%.'))
else:
    resultados.append(('Nível de combustível (RP-1)', True, None))

if not (nivel_oxidante >= 80):
    resultados.append(('Nível de oxidante (LOX)', False, f'Oxidante insuficiente ({nivel_oxidante:.1f}%). Mín: 80%.'))
else:
    resultados.append(('Nível de oxidante (LOX)', True, None))

if not (200 <= pressao_tanque_rp1 <= 400):
    resultados.append(('Pressão do tanque de RP-1', False, f'Pressão fora do intervalo ({pressao_tanque_rp1:.2f} kPa). Esperado: 200–400 kPa.'))
else:
    resultados.append(('Pressão do tanque de RP-1', True, None))

if not (300 <= pressao_tanque_lox <= 500):
    resultados.append(('Pressão do tanque de LOX', False, f'Pressão fora do intervalo ({pressao_tanque_lox:.2f} kPa). Esperado: 300–500 kPa.'))
else:
    resultados.append(('Pressão do tanque de LOX', True, None))

if not (integridade == 1):
    resultados.append(('Integridade estrutural', False, 'Falha estrutural detectada.'))
else:
    resultados.append(('Integridade estrutural', True, None))

status_modulos = all(passou for _, passou, _ in resultados)
resultados.append(('Status dos módulos', status_modulos, 'Módulos com falha detectada.'))

for nome, passou, erro in resultados:
    if not passou:
        print(f'  >> {nome}: FALHA')
        print(f'     {erro}')
        print('\n❌ DECOLAGEM ABORTADA')
        break
    print(f'  >> {nome}: OK')
    time.sleep(0.7)
else:
    print('\n✅ PRONTO PARA DECOLAR')

## 5. Análise IA — Gemini

Envia os dados de telemetria ao modelo **Gemini 2.5 Flash** (Google AI) para uma avaliação qualitativa independente.

**O que acontece nesta célula:**

O código monta um **prompt estruturado** com todos os valores inseridos na célula 3 — incluindo o gradiente térmico calculado e o status geral resultante da verificação. Esse prompt é enviado ao Gemini via `cliente.models.generate_content()`.

O modelo retorna uma análise em texto livre em português com três seções:
1. **Classificação por parâmetro** — cada dado recebe um nível: *normal*, *atenção* ou *crítico*, com base no contexto aeroespacial do prompt.
2. **Anomalias identificadas** — combinações de valores que, mesmo individualmente dentro do limite, podem indicar risco em conjunto.
3. **Sugestões de risco** — recomendações de ação antes do lançamento.

**Por que Gemini 2.5 Flash?**

O modelo `gemini-2.5-flash` é otimizado para respostas rápidas com boa capacidade de raciocínio — adequado para análises de telemetria onde velocidade e clareza importam mais do que geração criativa. O `textwrap.fill(..., width=150)` garante que linhas longas sejam quebradas para caber no terminal sem cortar palavras.

> A análise da IA é complementar, não substitui as verificações automáticas da célula 4. Uma decolagem abortada na célula 4 ainda passa pela análise da IA para que o modelo possa sugerir ações corretivas.

In [ ]:
# ANÁLISE IA
import textwrap

status_texto = 'PRONTO PARA DECOLAR' if status_modulos else 'DECOLAGEM ABORTADA'
integridade_texto = 'OK' if integridade == 1 else 'ERRO'

prompt = f"""
Você é um sistema de análise de telemetria aeroespacial. Analise os dados da Missão Aurora Siger:

TELEMETRIA:
- Temperatura interna: {temp_interna}°C
- Temperatura externa: {temp_externa}°C
- Gradiente térmico: {gradiente_termico:.1f}°C
- Nível de energia: {nivel_energia:.1f}%
- Nível de combustível (RP-1): {nivel_combustivel:.1f}%
- Nível de oxidante (LOX): {nivel_oxidante:.1f}%
- Pressão do tanque de RP-1: {pressao_tanque_rp1:.2f} kPa
- Pressão do tanque de LOX: {pressao_tanque_lox:.2f} kPa
- Integridade estrutural: {integridade_texto}
- Status geral: {status_texto}

Responda em português com:
1. Classificação de cada dado (normal / atenção / crítico)
2. Possíveis anomalias identificadas
3. Sugestões de risco
4. Formato de resposta: Texto simples, direto e claro. Sem formatação adicional.
"""

print('[ Análise IA - Gemini ]')
print('\n[ Aguardando resultado... ]\n')
resposta = cliente.models.generate_content(model='gemini-2.5-flash', contents=prompt)

for linha in resposta.text.splitlines():
    print(textwrap.fill(linha, width=150) if linha.strip() else '')
